# Using GPErks

In [1]:
import numpy as np
import torch
from GPErks_modified.log.logger import get_logger
from GPErks_modified.utils.random import set_seed
from sklearn.model_selection import train_test_split
from GPErks_modified.gp.data.dataset import Dataset
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.means import LinearMean
from gpytorch.kernels import RBFKernel, ScaleKernel
from torchmetrics import MeanSquaredError, R2Score
from GPErks_modified.gp.experiment import GPExperiment
from GPErks_modified.perks.cross_validation import KFoldCrossValidation
from GPErks_modified.train.early_stop import GLEarlyStoppingCriterion
from GPErks_modified.train.emulator import GPEmulator
from GPErks_modified.train.early_stop import NoEarlyStoppingCriterion

log = get_logger()
seed = 8
set_seed(seed)

/home/croderog/Desktop/IC_projects/simulation_toolbox/venv_gperks/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

# load dataset
mesh = 2
scenario = 41
basefolder=f"/media/croderog/SeagateExpansionDrive/HCM/{mesh}/scenarios/{scenario}"
emulators_folder_base = F"{basefolder}/output/emulators/"
feature_idx = 0


X_all =  np.loadtxt(f"{basefolder}/data/X.txt", dtype=float)
mask =  np.loadtxt(f"{basefolder}/output/output_mask.txt", dtype=float)
mask = mask.astype(bool)


input_masked = X_all[:mask.shape[0]]  # Trim X_all to match the size of the mask
X_ = input_masked[mask]
y_all = np.loadtxt(f"{basefolder}/data/Y.txt", dtype=float)

y_ = y_all[:,feature_idx]

with open(f"{basefolder}/data/xlabels.txt", "r") as f:
        x_labels = f.read().splitlines()

with open(f"{basefolder}/data/ylabels.txt", "r") as f:
        y_labels = f.read().splitlines()

emulators_folder = f"{emulators_folder_base}/{y_labels[feature_idx]}"

X_train = np.loadtxt(f"{emulators_folder}/X_train.txt", dtype=np.float64)
y_train = np.loadtxt(f"{emulators_folder}/y_train.txt", dtype=np.float64)

config_file = f"{emulators_folder}/emulator.ini"


In [3]:
from GPErks_modified.gp.experiment import load_experiment_from_config_file
from GPErks_modified.serialization.path import posix_path


dataset = Dataset(
                    X_train,
                    y_train,
                    x_labels=x_labels,
                    y_label=y_labels[feature_idx]
                    )
                
experiment = load_experiment_from_config_file(
            config_file,
            dataset  # notice that we still need to provide the dataset used!
            )
            
device = "cpu"
            
# loading emulator
best_model_file = posix_path(
    emulators_folder,
    "best_model.pth"
)
best_model_state = torch.load(best_model_file, map_location=torch.device(device))

emul = GPEmulator(experiment, device)
emul.model.load_state_dict(best_model_state)

<All keys matched successfully>

In [4]:
# Saltelli method for Sobol' indexes (Si) estimates
from GPErks_modified.perks.gsa import SobolGSA

gsa = SobolGSA(dataset, n=1024, seed=seed)

In [5]:
# estimate Si using the emulator
gsa = SobolGSA(dataset, n=1024, seed=seed)
gsa.estimate_Sobol_indices_with_emulator(emul, n_draws=1000)
gsa.summary()

Assembling...
Sampling...
Estimating..


100%|██████████| 1000/1000 [03:45<00:00,  4.44it/s]

                     STi
a_ventricles    0.649431
bf_ventricles   0.092374
bfs_ventricles  0.013644
bt_ventricles   0.226394
a_atria         0.000000
bf_atria        0.000000
bfs_atria       0.000000
bt_atria        0.000000
k_peri          0.067878
a_lvrv          0.000000
                      Si
a_ventricles    0.584966
bf_ventricles   0.063346
bfs_ventricles  0.011568
bt_ventricles   0.189867
a_atria         0.000000
bf_atria        0.000000
bfs_atria       0.000000
bt_atria        0.000000
k_peri          0.042610
a_lvrv          0.000000
                                      Sij
(a_ventricles, bf_ventricles)    0.024824
(a_ventricles, bfs_ventricles)   0.015209
(a_ventricles, bt_ventricles)    0.025205
(a_ventricles, a_atria)          0.015233
(a_ventricles, bf_atria)         0.015521
(a_ventricles, bfs_atria)        0.015112
(a_ventricles, bt_atria)         0.015508
(a_ventricles, k_peri)           0.020930
(a_ventricles, a_lvrv)           0.015428
(bf_ventricles, bfs_ventricles

In [6]:
gsa.correct_Sobol_indices()

In [11]:
import pandas as pd
df_STi = pd.DataFrame(
    data=np.round(np.median(gsa.ST, axis=0), 6).reshape(-1, 1),
    index=gsa.index_i,
    columns=["STi"],
)
df_Si = pd.DataFrame(
    data=np.round(np.median(gsa.S1, axis=0), 6).reshape(-1, 1),
    index=gsa.index_i,
    columns=["Si"],
)
df_Sij = pd.DataFrame(
    data=np.round(np.median(gsa.S2, axis=0), 6).reshape(-1, 1),
    index=[
        "(" + elem[0] + ", " + elem[1] + ")" for elem in gsa.index_ij
    ],
    columns=["Sij"],
)

# Save df_STi to a CSV file
df_STi.to_csv(f'{emulators_folder}/df_STi.csv', index=True)

# Save df_Si to a CSV file
df_Si.to_csv(f'{emulators_folder}/df_Si.csv', index=True)

# Save df_Sij to a CSV file
df_Sij.to_csv(f'{emulators_folder}/df_Sij.csv', index=True)
